========================================
### GOLD LAYER – PRODUCT / SELLER METRICS
----------------------------------------
###Source : Silver Order Items
###Target : Delta Gold Aggregate Table
###Grain  : One row per product per seller
###Load   : Full Refresh (Overwrite)
###Purpose: Product & seller performance
========================================


In [0]:
order_items = spark.table("olist_silver_order_items")


### AGGREGATE 

In [0]:
from pyspark.sql.functions import countDistinct, sum, avg

gold_product = (
    order_items
    .groupBy("product_id", "seller_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("price").alias("total_revenue"),
        avg("price").alias("avg_price"),
        sum("freight_value").alias("total_freight")
    )
)


###WRITE TO GOLD

In [0]:
(
    gold_product
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_gold_product_metrics")
)


### OPTIMIZE

In [0]:
%sql
OPTIMIZE olist_gold_product_metrics
ZORDER BY (seller_id);


path,metrics
abfss://unity-catalog-storage@dbstoragetic7vxegr5zes.dfs.core.windows.net/7405614582366842/__unitystorage/catalogs/4444e7c2-d2e1-4e77-b5c3-b026efd7d282/tables/7c2bab5b-d8d0-4f92-9d8d-2f5998a8f7bd,"List(1, 4, List(993237, 993237, 993237.0, 1, 993237), List(278959, 301401, 287868.25, 4, 1151473), 0, List(minCubeSize(107374182400), List(0, 0), List(4, 1151473), 0, List(4, 1151473), 1, null), null, 0, 1, 4, 0, false, 0, 0, 1769886643621, 1769886645394, 4, 1, null, List(0, 0), null, 6, 6, 345, 0, null)"


In [0]:
%sql
SELECT COUNT(*) FROM olist_gold_product_metrics;


count(1)
34448
